In [1]:
from cxrmate_ed.configuration_cxrmate_ed import CXRMateEDConfig
from cxrmate_ed.modelling_cxrmate_ed import CXRMateEDModel
from cxrmate_ed.lightning_module import SCSTSectionsWeightedCXRBERTBERTScoreARNReward
import torch
import transformers
import warnings
import yaml
from pathlib import Path
import os
from huggingface_hub import HfApi


/apps/python/3.12.0/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/apps/python/3.12.0/lib/python3.12/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/apps/python/3.12.0/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/apps/python/3.12.0/lib/python3.12/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


In [2]:
transformers.__version__, torch.__version__

('4.46.3', '2.1.1')

In [3]:
# Hub checkpoint name:
hub_ckpt_name = 'aehrc/cxrmate-ed'

In [4]:
# Paths:
ckpt_zoo_dir = '/datasets/work/hb-mlaifsp-mm/work/checkpoints'
database_dir = '/scratch3/nic261/database/cxrmate_ed'
ckpt_path = '/datasets/work/hb-mlaifsp-mm/work/experiments/cxrmate_ed/rl/003_scst_sections_weighted_cxrbert_bertscore_ngram/trial_0/epoch=31-step=76416-val_findings_bertscore_f1=0.459060.ckpt'
args_path = '/datasets/work/hb-mlaifsp-mm/work/experiments/cxrmate_ed/rl/003_scst_sections_weighted_cxrbert_bertscore_ngram/trial_0/arguments/session_6.yaml'

In [5]:
transformers.AutoConfig.register('cxrmate-ed', CXRMateEDConfig)
transformers.AutoModelForCausalLM.register(CXRMateEDConfig, CXRMateEDModel)

In [6]:
CXRMateEDConfig.register_for_auto_class()
CXRMateEDModel.register_for_auto_class('AutoModelForCausalLM')

In [7]:
# HF Model and tokenizer:
args = yaml.safe_load(Path(args_path).read_text())
lightning_module = SCSTSectionsWeightedCXRBERTBERTScoreARNReward(warm_start_modules=False, **args)
model = lightning_module.model
tokenizer = lightning_module.tokenizer

Description, Special token, Index
bos_token, [BOS], 1
eos_token, [EOS], 2
unk_token, [UNK], 0
sep_token, [SEP], 3
pad_token, [PAD], 4
cls_token, [BOS], 1
mask_token, [MASK], 5


In [8]:
# Rename state dict keys:
state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=True)['state_dict']

for key in list(state_dict.keys()):
    if 'encoder_decoder.' in key:
        state_dict[key.replace('encoder_decoder.', '')] = state_dict.pop(key)

for key in list(state_dict.keys()):
    if 'encoder.uniformer.' in key:
        state_dict[key.replace('encoder.uniformer.', 'image_encoder.encoder.uniformer.')] = state_dict.pop(key)

for key in list(state_dict.keys()):
    if 'decoder.model.' in key:
        state_dict[key.replace('decoder.model.', 'language_model.model.')] = state_dict.pop(key)

for key in list(state_dict.keys()):
    if 'decoder.lm_head.weight' in key:
        state_dict[key.replace('decoder.lm_head.weight', 'language_model.lm_head.weight')] = state_dict.pop(key)

for key in list(state_dict.keys()):
    if 'encoder.projection_head.' in key:
        state_dict[key.replace('encoder.projection_head.', 'image_encoder.adapter.')] = state_dict.pop(key)

/apps/pytorch/2.1.1-py312-cu122-mpi/lib/python3.12/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [9]:
model.load_state_dict(state_dict)

<All keys matched successfully>

In [10]:
# Save model:
save_path = '/scratch3/nic261/checkpoints/cxrmate_ed'
model.save_pretrained(save_path)

[2024-12-13 11:47:45,678] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/nic261/.local/lib/python3.12/site-packages/pydantic/_internal/_fields.py:160: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/nic261/.local/lib/python3.12/site-packages/pydantic/_internal/_config.py:334: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [11]:
# Push to hub:
model.push_to_hub(hub_ckpt_name)
tokenizer.push_to_hub(hub_ckpt_name)

README.md:   0%|          | 0.00/5.26k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/aehrc/cxrmate-ed/commit/d92f6a0fbdd4a81c9b8727d2da625ba585d4056f', commit_message='Upload tokenizer', commit_description='', oid='d92f6a0fbdd4a81c9b8727d2da625ba585d4056f', pr_url=None, pr_revision=None, pr_num=None)

In [12]:
api = HfApi()

In [13]:
api.upload_file(
    path_or_fileobj='cxrmate_ed/lookup_tables.json',
    path_in_repo='lookup_tables.json',
    repo_id=hub_ckpt_name,
)
api.upload_file(
    path_or_fileobj='cxrmate_ed/tables.json',
    path_in_repo='tables.json',
    repo_id=hub_ckpt_name,
)
api.upload_file(
    path_or_fileobj='cxrmate_ed/token_type_ids.json',
    path_in_repo='token_type_ids.json',
    repo_id=hub_ckpt_name,
)
api.upload_file(
    path_or_fileobj='cxrmate_ed/mimic_cxr_jpg_train_study_ids.json',
    path_in_repo='mimic_cxr_jpg_train_study_ids.json',
    repo_id=hub_ckpt_name,
)
api.upload_file(
    path_or_fileobj='cxrmate_ed/mimic_cxr_jpg_validate_study_ids.json',
    path_in_repo='mimic_cxr_jpg_validate_study_ids.json',
    repo_id=hub_ckpt_name,
)
api.upload_file(
    path_or_fileobj='cxrmate_ed/mimic_cxr_jpg_test_study_ids.json',
    path_in_repo='mimic_cxr_jpg_test_study_ids.json',
    repo_id=hub_ckpt_name,
)
api.upload_file(
    path_or_fileobj='cxrmate_ed/mimic_iv_ed_mimic_cxr_jpg_train_study_ids.json',
    path_in_repo='mimic_iv_ed_mimic_cxr_jpg_train_study_ids.json',
    repo_id=hub_ckpt_name,
)
api.upload_file(
    path_or_fileobj='cxrmate_ed/mimic_iv_ed_mimic_cxr_jpg_validate_study_ids.json',
    path_in_repo='mimic_iv_ed_mimic_cxr_jpg_validate_study_ids.json',
    repo_id=hub_ckpt_name,
)
api.upload_file(
    path_or_fileobj='cxrmate_ed/mimic_iv_ed_mimic_cxr_jpg_test_study_ids.json',
    path_in_repo='mimic_iv_ed_mimic_cxr_jpg_test_study_ids.json',
    repo_id=hub_ckpt_name,
)

CommitInfo(commit_url='https://huggingface.co/aehrc/cxrmate-ed/commit/88235a43d3f0f45599c60ce99770c5b70f475ebe', commit_message='Upload mimic_iv_ed_mimic_cxr_jpg_test_study_ids.json with huggingface_hub', commit_description='', oid='88235a43d3f0f45599c60ce99770c5b70f475ebe', pr_url=None, pr_revision=None, pr_num=None)